# 02 内容视图构建

**目的**：对下载的表格进行剖析，生成列描述，使用 Sentence-Transformer 编码，
构建 `Z_tabcontent` 数据集向量和 `S_tabcontent` 相似度图，计算元数据-内容一致性。

实现 CONTENT_VIEW_EXTENSION.md §3（内容视图构建）和 §4（一致性分析）。

**输出**：
- `tmp/content/col_profiles.parquet` — 列级剖析统计
- `tmp/content/col_descriptions.parquet` — 模板生成的列描述
- `tmp/content/col_embeddings.parquet` — Sentence-Transformer 列嵌入
- `tmp/content/Z_tabcontent.parquet` — 聚合后的数据集向量
- `tmp/content/S_tabcontent_symrow_k50_manifest.json` + 分区文件 — 内容相似度图
- `tmp/content/consistency_meta_content.parquet` — 元数据-内容一致性分数

### 配置与导入

**功能**：导入所需库，定义路径常量、剖析参数、嵌入参数和相似度图参数

**输入**：
- 无外部输入（硬编码配置）

**输出**：
- `ROOT` / `CONTENT_DIR` / `TAB_RAW_DIR` 等路径变量
- `MAX_ROWS` / `MAX_COLS` / `EMBED_MODEL` / `K_SIM` 等参数常量

In [ ]:
# ---- 配置 ----
import os, sys, json, warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any

import numpy as np
import pandas as pd
from scipy import sparse

warnings.filterwarnings("ignore", category=FutureWarning)

# 路径配置（notebook 位于 notebooks/04_content/，项目根目录在两级之上）
ROOT        = Path(".").resolve().parent.parent
TMP_DIR     = ROOT / "tmp"
CONTENT_DIR = TMP_DIR / "content"
TAB_RAW_DIR = ROOT / "data" / "tabular_raw"

# 确保 src 可导入
sys.path.insert(0, str(ROOT))
from src.content.sampling import (
    read_by_ext, sample_table, profile_column, profile_table, col_to_description, ColStats
)
from src.content.encoding import aggregate_dataset_vector, W_TYPE
from src.content.similarity import (
    sym_and_rownorm, save_partitioned_edges,
    load_manifest_flexible, load_edges_from_manifest, build_neighbor_dict,
)
from src.content.consistency import compute_jaccard_and_consistency

# 剖析参数
MAX_ROWS  = 1024
MAX_COLS  = 60

# 嵌入参数
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM   = 384

# 相似度图参数
K_SIM      = 50
N_TOTAL    = 521735
PART_SIZE  = 2_000_000

print(f"MAX_ROWS={MAX_ROWS}, MAX_COLS={MAX_COLS}, K_SIM={K_SIM}")
print(f"EMBED_MODEL={EMBED_MODEL}, EMBED_DIM={EMBED_DIM}")

### 按扩展名读取表格文件

**功能**：根据文件扩展名分发到对应的 pandas 读取函数，支持 csv/tsv/parquet/xlsx/xls 格式，含编码回退机制

**输入**：
- `path` — 表格文件路径
- `nrows`（可选）— 最大读取行数

**输出**：
- 返回 `pd.DataFrame` 或读取失败时返回 `None`

In [ ]:
# read_by_ext is imported from src.content.sampling.
# See src/content/sampling.py for the implementation.
print("Using read_by_ext from src.content.sampling")

### 低成本表格采样

**功能**：读取并采样表格，丢弃全空列、常量列和 ID 类列，按信息分数保留前 max_cols 列

**输入**：
- `path` — 表格文件路径
- `max_rows` — 最大采样行数（默认 1024）
- `max_cols` — 最大保留列数（默认 60）

**输出**：
- 返回采样后的 `pd.DataFrame` 或失败时返回 `None`

In [ ]:
# sample_table is imported from src.content.sampling.
# See src/content/sampling.py for the implementation.
print("Using sample_table from src.content.sampling")

### 列统计数据类 + 列类型识别

**功能**：定义 `ColStats` 数据类存储列级统计信息；`profile_column` 函数按优先级（数值 > 日期 > 文本 > 类别）自动识别列类型并计算类型相关统计量

**输入**：
- `series` — pandas Series（单列数据）
- `col_name` — 列名字符串

**输出**：
- `ColStats` 实例 — 包含列名、类型、缺失率、唯一值数、类型相关统计

In [ ]:
# ColStats and profile_column are imported from src.content.sampling.
# See src/content/sampling.py for the implementation.
print("Using ColStats, profile_column from src.content.sampling")

### 整表列剖析

**功能**：对 DataFrame 中的所有列逐一调用 `profile_column` 进行剖析

**输入**：
- `df` — 待剖析的 DataFrame

**输出**：
- `List[ColStats]` — 每列的剖析结果列表

In [ ]:
# profile_table is imported from src.content.sampling.
# See src/content/sampling.py for the implementation.
print("Using profile_table from src.content.sampling")

### 列描述文本生成

**功能**：根据 ColStats 使用模板生成英文列描述文本，四种类型各有对应模板（遵循 §3.3 规范）

**输入**：
- `cs` — `ColStats` 实例

**输出**：
- 返回英文描述字符串（用于后续嵌入编码）

In [ ]:
# col_to_description is imported from src.content.sampling.
# See src/content/sampling.py for the implementation.
print("Using col_to_description from src.content.sampling")

### 加载内容子集与主表注册表

**功能**：加载 D_content 子集和主表注册表，合并两者并筛选出有实际表格文件的数据集

**输入**：
- `tmp/content/d_content.parquet` — D_content 子集
- `tmp/content/main_tables.parquet` — 主表注册表

**输出**：
- `work` DataFrame — 合并后的工作集（仅含有表格文件的行）

In [ ]:
# 加载 d_content 和 main_tables
d_content = pd.read_parquet(CONTENT_DIR / "d_content.parquet", engine="fastparquet")
main_tables = pd.read_parquet(CONTENT_DIR / "main_tables.parquet", engine="fastparquet")
print(f"d_content: {len(d_content)} datasets")
print(f"main_tables: {len(main_tables)} entries, "
      f"{(main_tables['main_table_path'] != '').sum()} with tables")

# 合并：d_content 用 Id，main_tables 用 DatasetId，两者等价
mt_ren = main_tables.rename(columns={"DatasetId": "Id"})
work = d_content.merge(mt_ren, on=["Id", "doc_idx"], how="inner")

# 筛选有实际表格文件的数据集
work = work[work["main_table_path"] != ""].copy()
print(f"Datasets with tables to profile: {len(work)}")

### 批量剖析循环

**功能**：遍历所有有表格文件的数据集，依次采样→剖析→生成描述，收集列级统计和描述文本

**输入**：
- `work` DataFrame — 含有 `main_table_path` 的工作集

**输出**：
- `tmp/content/col_profiles.parquet` — 列级剖析统计
- `tmp/content/col_descriptions.parquet` — 列描述文本

In [ ]:
# 批量剖析循环
all_profiles = []    # 列剖析结果字典列表
all_descriptions = [] # 列描述字典列表

n_success = 0
n_fail = 0

for i, (_, row) in enumerate(work.iterrows()):
    ds_id = int(row["Id"])
    doc_idx = int(row["doc_idx"])
    table_path = row["main_table_path"]

    # 采样表格
    df = sample_table(table_path, MAX_ROWS, MAX_COLS)
    if df is None or df.empty:
        n_fail += 1
        continue

    # 剖析各列
    profiles = profile_table(df)
    n_success += 1

    for cs in profiles:
        desc = col_to_description(cs)
        all_profiles.append({
            "DatasetId": ds_id,
            "doc_idx": doc_idx,
            "col_name": cs.name,
            "dtype": cs.dtype,
            "missing_pct": cs.missing_pct,
            "n_unique": cs.n_unique,
            "unique_pct": cs.unique_pct,
        })
        all_descriptions.append({
            "DatasetId": ds_id,
            "doc_idx": doc_idx,
            "col_name": cs.name,
            "description": desc,
            "dtype": cs.dtype,
            "missing_pct": cs.missing_pct,
            "unique_pct": cs.unique_pct,
        })

    if (i + 1) % 100 == 0:
        print(f"  Profiled {i+1}/{len(work)} datasets "
              f"(success={n_success}, fail={n_fail}, cols so far={len(all_profiles)})")

# 保存
col_profiles_df = pd.DataFrame(all_profiles)
col_descriptions_df = pd.DataFrame(all_descriptions)

col_profiles_df.to_parquet(CONTENT_DIR / "col_profiles.parquet", engine="fastparquet")
col_descriptions_df.to_parquet(CONTENT_DIR / "col_descriptions.parquet", engine="fastparquet")

print(f"\nProfiling complete:")
print(f"  Datasets profiled: {n_success} (failed: {n_fail})")
print(f"  Total columns: {len(col_profiles_df)}")
print(f"  Type distribution:")
print(col_profiles_df["dtype"].value_counts().to_string())

### Sentence-Transformer 编码

**功能**：使用预训练的 Sentence-Transformer 模型将所有列描述文本编码为归一化向量；若模型不可用则使用随机占位向量

**输入**：
- `col_descriptions_df` — 列描述 DataFrame
- `EMBED_MODEL` — 模型名称

**输出**：
- `embeddings` — (N_cols, 384) 归一化嵌入矩阵
- `tmp/content/col_embeddings.parquet` — 持久化的列嵌入

In [ ]:
# Sentence-Transformer 编码
try:
    from sentence_transformers import SentenceTransformer
    ST_AVAILABLE = True
except ImportError:
    ST_AVAILABLE = False
    print("sentence-transformers not installed. Install via: pip install sentence-transformers")
    print("Encoding will be skipped; downstream cells will use random placeholders.")

descriptions = col_descriptions_df["description"].tolist()
print(f"Descriptions to encode: {len(descriptions)}")

if ST_AVAILABLE:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Encoding on device: {device}")

    model = SentenceTransformer(EMBED_MODEL, device=device)
    embeddings = model.encode(
        descriptions,
        batch_size=256,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    embeddings = np.array(embeddings, dtype=np.float32)
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
else:
    # 占位：随机单位向量用于下游流水线测试
    print("Using random placeholder embeddings for pipeline testing.")
    rng = np.random.RandomState(42)
    embeddings = rng.randn(len(descriptions), EMBED_DIM).astype(np.float32)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings = embeddings / np.maximum(norms, 1e-12)

print(f"Embeddings shape: {embeddings.shape}")

# 保存列嵌入
emb_df = col_descriptions_df[["DatasetId", "doc_idx", "col_name"]].copy()
for j in range(EMBED_DIM):
    emb_df[f"e{j}"] = embeddings[:, j]
emb_df.to_parquet(CONTENT_DIR / "col_embeddings.parquet", engine="fastparquet")
print(f"Saved col_embeddings.parquet: {emb_df.shape}")

### 数据集向量聚合函数

**功能**：将单个数据集内的多列嵌入加权聚合为一个数据集级向量，权重由列类型、缺失率和唯一率决定

**输入**：
- `col_embs` — (N_cols, EMBED_DIM) 列嵌入矩阵
- `col_stats_list` — 每列的 dtype/missing_pct/unique_pct 字典列表

**输出**：
- 返回 L2 归一化的 (EMBED_DIM,) 数据集向量

In [ ]:
# aggregate_dataset_vector and W_TYPE are imported from src.content.encoding.
# See src/content/encoding.py for the implementation.
print(f"Using aggregate_dataset_vector from src.content.encoding, W_TYPE={W_TYPE}")

### 构建 Z_tabcontent 矩阵

**功能**：按 DatasetId 分组，对每个数据集的列嵌入调用聚合函数，堆叠为全局数据集向量矩阵

**输入**：
- `tmp/content/col_embeddings.parquet` — 列嵌入
- `tmp/content/col_profiles.parquet` — 列剖析统计

**输出**：
- `Z_df` DataFrame — (B_ds, 384) 数据集向量矩阵
- `tmp/content/Z_tabcontent.parquet` — 持久化的向量矩阵

In [ ]:
# 构建 Z_tabcontent：按 DatasetId 分组聚合并堆叠
emb_cols = [f"e{j}" for j in range(EMBED_DIM)]

# 重新加载列嵌入和剖析结果
emb_df = pd.read_parquet(CONTENT_DIR / "col_embeddings.parquet", engine="fastparquet")
prof_df = pd.read_parquet(CONTENT_DIR / "col_profiles.parquet", engine="fastparquet")

# 合并剖析信息用于权重计算
merged = emb_df.merge(
    prof_df[["DatasetId", "col_name", "dtype", "missing_pct", "unique_pct"]],
    on=["DatasetId", "col_name"],
    how="left",
    suffixes=("", "_prof")
)

# 使用正确的 dtype/missing/unique 列
dtype_col = "dtype_prof" if "dtype_prof" in merged.columns else "dtype"
miss_col = "missing_pct_prof" if "missing_pct_prof" in merged.columns else "missing_pct"
uniq_col = "unique_pct_prof" if "unique_pct_prof" in merged.columns else "unique_pct"

dataset_ids = merged["DatasetId"].unique()
z_rows = []

for ds_id in dataset_ids:
    mask = merged["DatasetId"] == ds_id
    sub = merged[mask]
    doc_idx = int(sub["doc_idx"].iloc[0])

    col_embs = sub[emb_cols].values.astype(np.float32)
    col_stats = [
        {"dtype": r[dtype_col], "missing_pct": r[miss_col], "unique_pct": r[uniq_col]}
        for _, r in sub.iterrows()
    ]

    z = aggregate_dataset_vector(col_embs, col_stats)
    row_dict = {"doc_idx": doc_idx}
    for j in range(EMBED_DIM):
        row_dict[f"f{j}"] = float(z[j])
    z_rows.append(row_dict)

Z_df = pd.DataFrame(z_rows)
Z_df["doc_idx"] = Z_df["doc_idx"].astype(int)
Z_df = Z_df.sort_values("doc_idx").reset_index(drop=True)
Z_df.to_parquet(CONTENT_DIR / "Z_tabcontent.parquet", engine="fastparquet")

print(f"Z_tabcontent.parquet: {Z_df.shape}")
print(f"  doc_idx range: [{Z_df['doc_idx'].min()}, {Z_df['doc_idx'].max()}]")

# 验证 L2 范数
feat_cols = [f"f{j}" for j in range(EMBED_DIM)]
norms = np.linalg.norm(Z_df[feat_cols].values, axis=1)
print(f"  L2 norms: min={norms.min():.4f}, max={norms.max():.4f}, mean={norms.mean():.4f}")

### FAISS 近邻检索构建内容相似图

**功能**：使用 FAISS IndexFlatIP 对数据集向量执行 top-K 内积近邻搜索，收集 COO 稀疏矩阵三元组

**输入**：
- `tmp/content/Z_tabcontent.parquet` — 数据集向量矩阵
- `K_SIM` — 近邻数量

**输出**：
- `rows_arr` / `cols_arr` / `vals_arr` — COO 格式的边三元组

In [ ]:
# FAISS IndexFlatIP：构建内容相似度图
try:
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    FAISS_AVAILABLE = False
    print("faiss not installed. Install via: pip install faiss-cpu (or faiss-gpu)")

# 加载 Z_tabcontent
Z_df = pd.read_parquet(CONTENT_DIR / "Z_tabcontent.parquet", engine="fastparquet")
doc_indices = Z_df["doc_idx"].values.astype(np.int64)
feat_cols = [f"f{j}" for j in range(EMBED_DIM)]
Z = Z_df[feat_cols].values.astype(np.float32)

# L2 归一化（应已归一化，此处确保）
norms = np.linalg.norm(Z, axis=1, keepdims=True)
Z = Z / np.maximum(norms, 1e-12)
Z = np.ascontiguousarray(Z)

B_actual = len(Z)
print(f"Z shape: {Z.shape}, B_actual={B_actual}")

if FAISS_AVAILABLE:
    # 构建 IndexFlatIP
    index = faiss.IndexFlatIP(EMBED_DIM)

    # 尝试使用 GPU
    try:
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
        print("Using FAISS GPU index")
    except Exception:
        print("Using FAISS CPU index")

    index.add(Z)

    # 搜索 top-(K+1) 以排除自身
    k_search = min(K_SIM + 1, B_actual)
    scores, idxs = index.search(Z, k_search)
    print(f"FAISS search done: scores shape={scores.shape}")
else:
    # 回退：使用 numpy 暴力计算（较慢）
    print("Computing similarities with numpy (slow fallback)...")
    sim_matrix = Z @ Z.T
    k_search = min(K_SIM + 1, B_actual)
    idxs = np.argsort(-sim_matrix, axis=1)[:, :k_search]
    scores = np.take_along_axis(sim_matrix, idxs, axis=1)
    print(f"Numpy similarity done: scores shape={scores.shape}")

# 将局部索引转换为全局 doc_idx，收集 COO 三元组
rows_list = []
cols_list = []
vals_list = []

for i in range(B_actual):
    global_i = int(doc_indices[i])
    for j_pos in range(k_search):
        local_j = int(idxs[i, j_pos])
        if local_j == i:  # 排除自身
            continue
        global_j = int(doc_indices[local_j])
        val = float(scores[i, j_pos])
        if val > 0:
            rows_list.append(global_i)
            cols_list.append(global_j)
            vals_list.append(val)

rows_arr = np.array(rows_list, dtype=np.int64)
cols_arr = np.array(cols_list, dtype=np.int64)
vals_arr = np.array(vals_list, dtype=np.float32)
print(f"COO triplets: {len(rows_arr)} edges")

### 对称化与行归一化

**功能**：对稀疏矩阵执行 max 对称化，移除对角线，然后进行 L1 行归一化使其成为行随机矩阵

**输入**：
- `rows` / `cols` / `vals` — COO 格式的边数据
- `N` — 矩阵维度

**输出**：
- 返回对称化并行归一化后的 `(coo_rows, coo_cols, coo_vals)` 三元组

In [ ]:
# sym_and_rownorm is imported from src.content.similarity.
# See src/content/similarity.py for the implementation.
print("Using sym_and_rownorm from src.content.similarity")

### 分区存储 COO 边 + manifest

**功能**：将 COO 稀疏矩阵按分区大小切分为多个 parquet 文件，并生成 manifest.json 索引文件

**输入**：
- `rows` / `cols` / `vals` — COO 格式的边数据
- `N` / `prefix` / `k` / `output_dir` — 矩阵参数和输出配置

**输出**：
- `{prefix}_k{k}_part*.parquet` — 分区边文件
- `{prefix}_k{k}_manifest.json` — 索引清单

In [ ]:
# save_partitioned_edges is imported from src.content.similarity.
# See src/content/similarity.py for the implementation.
print("Using save_partitioned_edges from src.content.similarity")

### 执行对称化与保存

**功能**：对 FAISS 检索得到的原始 COO 边执行对称化和行归一化，然后分区保存为内容相似度图

**输入**：
- `rows_arr` / `cols_arr` / `vals_arr` — FAISS 检索的原始 COO 边
- `N_TOTAL` — 全局节点数

**输出**：
- `tmp/content/S_tabcontent_symrow_k50_manifest.json` + 分区文件

In [ ]:
# 执行对称化和行归一化，然后保存
sym_rows, sym_cols, sym_vals = sym_and_rownorm(rows_arr, cols_arr, vals_arr, N_TOTAL)
print(f"After sym+rownorm: {len(sym_rows)} edges")

save_partitioned_edges(
    sym_rows, sym_cols, sym_vals, N_TOTAL,
    prefix="S_tabcontent_symrow", k=K_SIM,
    output_dir=CONTENT_DIR,
    note="row-stochastic; content view from tabular column embeddings; D_content subset only"
)

### Manifest 灵活加载 + 边加载 + 邻居构建

**功能**：定义三个工具函数——灵活加载 manifest JSON（兼容多种键名）、从 manifest 加载所有边分区、从边 DataFrame 构建邻居字典

**输入**：
- `prefix` / `base_dir` — 视图名称和基础目录

**输出**：
- `load_manifest_flexible` → `(manifest_dict, parts_list, parent_dir)`
- `load_edges_from_manifest` → `(edges_df, manifest_dict)`
- `build_neighbor_dict` → `{doc_idx: set(neighbor_indices)}`

In [ ]:
# load_manifest_flexible, load_edges_from_manifest, build_neighbor_dict
# are imported from src.content.similarity.
# See src/content/similarity.py for the implementation.
print("Using manifest/edge/neighbor utilities from src.content.similarity")

### 加载 S_fused3 邻居

**功能**：加载三视图融合相似度图的边数据，仅保留 D_content 文档的邻居关系

**输入**：
- `tmp/content/d_content.parquet` — D_content 子集
- `tmp/S_fused3_symrow_k50_manifest.json` + 分区文件

**输出**：
- `N_meta` — `{doc_idx: set(邻居)}` 元数据视图邻居字典

In [ ]:
# 仅为 D_content 文档加载 S_fused3 邻居
d_content = pd.read_parquet(CONTENT_DIR / "d_content.parquet", engine="fastparquet")
d_content_set = set(d_content["doc_idx"].astype(int).values)
print(f"D_content docs: {len(d_content_set)}")

print("Loading S_fused3 edges...")
fused3_edges, fused3_man = load_edges_from_manifest("S_fused3_symrow", TMP_DIR)
print(f"  Total edges: {len(fused3_edges)}")
N_meta = build_neighbor_dict(fused3_edges, d_content_set)
print(f"  Docs with meta neighbors: {len(N_meta)}")
del fused3_edges  # 释放内存

### 加载 S_tabcontent 邻居

**功能**：加载内容视图相似度图的边数据并构建邻居字典

**输入**：
- `tmp/content/S_tabcontent_symrow_k50_manifest.json` + 分区文件

**输出**：
- `N_cont` — `{doc_idx: set(邻居)}` 内容视图邻居字典

In [ ]:
# 加载 S_tabcontent 邻居
print("Loading S_tabcontent edges...")
cont_edges, cont_man = load_edges_from_manifest("S_tabcontent_symrow", CONTENT_DIR)
print(f"  Total edges: {len(cont_edges)}")
N_cont = build_neighbor_dict(cont_edges, d_content_set)
print(f"  Docs with content neighbors: {len(N_cont)}")

### 计算 Jaccard 重叠与加权一致性

**功能**：对 D_content 中每个文档计算元数据-内容邻居的 Jaccard 重叠度和加权一致性分数

**输入**：
- `d_content_ids` — D_content 文档索引列表
- `N_meta` / `N_cont` — 元数据和内容视图的邻居字典
- `cont_edges` / `fused3_dir`（可选）— 用于加权一致性的边权重

**输出**：
- `consistency_df` DataFrame — 每文档的 Jaccard 和加权一致性分数
- `tmp/content/consistency_meta_content.parquet` — 持久化的一致性分数

In [ ]:
# compute_jaccard_and_consistency is imported from src.content.consistency.
# See src/content/consistency.py for the implementation.

# 计算一致性
d_content_ids = sorted(d_content_set)
print(f"Computing consistency for {len(d_content_ids)} docs...")

consistency_df = compute_jaccard_and_consistency(
    d_content_ids, N_meta, N_cont,
    cont_edges_df=cont_edges,
    fused3_dir=TMP_DIR,
    k=K_SIM,
)

consistency_df.to_parquet(CONTENT_DIR / "consistency_meta_content.parquet", engine="fastparquet")
print(f"Saved consistency_meta_content.parquet: {len(consistency_df)} rows")
print(f"\nJaccard J(i):")
print(f"  mean={consistency_df['jaccard'].mean():.4f}, "
      f"median={consistency_df['jaccard'].median():.4f}, "
      f"std={consistency_df['jaccard'].std():.4f}")
print(f"Weighted consistency c(i):")
print(f"  mean={consistency_df['weighted_consistency'].mean():.4f}, "
      f"median={consistency_df['weighted_consistency'].median():.4f}, "
      f"std={consistency_df['weighted_consistency'].std():.4f}")

### 一致性可视化

**功能**：绘制 Jaccard 重叠度和加权一致性分数的直方图，标注均值线，并输出描述性统计

**输入**：
- `consistency_df` DataFrame — 一致性分数数据

**输出**：
- `tmp/content/fig_consistency_histograms.png` — 一致性分布直方图

In [ ]:
# 一致性可视化
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Jaccard 直方图
ax = axes[0]
ax.hist(consistency_df["jaccard"], bins=50, color="steelblue", edgecolor="white", alpha=0.8)
ax.axvline(consistency_df["jaccard"].mean(), color="red", linestyle="--",
           label=f"mean={consistency_df['jaccard'].mean():.3f}")
ax.set_xlabel("Jaccard J(i)")
ax.set_ylabel("Count")
ax.set_title("Meta-Content Jaccard Overlap")
ax.legend()

# 加权一致性直方图
ax = axes[1]
ax.hist(consistency_df["weighted_consistency"], bins=50, color="darkorange",
        edgecolor="white", alpha=0.8)
ax.axvline(consistency_df["weighted_consistency"].mean(), color="red", linestyle="--",
           label=f"mean={consistency_df['weighted_consistency'].mean():.3f}")
ax.set_xlabel("Weighted Consistency c(i)")
ax.set_ylabel("Count")
ax.set_title("Meta-Content Weighted Consistency")
ax.legend()

plt.tight_layout()
fig.savefig(CONTENT_DIR / "fig_consistency_histograms.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_consistency_histograms.png")

# 描述性统计表
print("\nSummary statistics:")
print(consistency_df[["jaccard", "weighted_consistency", "n_meta", "n_cont", "n_intersect"]].describe())

## 总结

**产出文件**：
- `tmp/content/col_profiles.parquet` — 列级剖析统计
- `tmp/content/col_descriptions.parquet` — 模板生成的列描述
- `tmp/content/col_embeddings.parquet` — Sentence-Transformer 列嵌入
- `tmp/content/Z_tabcontent.parquet` — 聚合后的数据集向量（B_ds × 384）
- `tmp/content/S_tabcontent_symrow_k50_manifest.json` + 分区文件 — 内容相似度图
- `tmp/content/consistency_meta_content.parquet` — 元数据-内容一致性分数

**下一步**：运行 `03_content_fusion_experiments.ipynb` 进行融合和评测。